# Linear regression — from scratch

*Companion to **Eps 19–23** of the Linear Regression series.*

We build the simplest model in ML, one concept at a time. By the end you'll have trained it three ways: by hand, in PyTorch, in sklearn.

## Chapter 1 · Setup

Just the imports. NumPy for the math, matplotlib for plots.

## Chapter 2 · The dataset

Seven cars. Each has a weight (in thousands of pounds) and an MPG. That's it.

Print it as a table so we can see the pattern.

Heavier cars get fewer MPG. The relationship is visible in the numbers. Now let's plot it.

The dots roughly follow a downward line. **A line is the simplest model that could capture this pattern.**

We want to find *the* line — the best one. Let's build up to that.

## Chapter 3 · The model: `y = w·x + b`

A line has two numbers that define it:

- **`w`** is the slope — how steep the line is.
- **`b`** is the intercept — where the line crosses the y-axis.

Pick any (w, b), get a line. Change either, get a different line.

Let's pick a starting guess. We'll come back and improve it. For now: `w = -3`, `b = 30`.

Predict the MPG of the first car (weight 2.4).

Compare that to the actual MPG (24). Off by a bit. Now do it for all 7 cars.

Side-by-side comparison:

Let's plot the line over the data and see how it looks.

The line is roughly in the right zone but not great. We need a way to *measure* how not-great it is.

## Chapter 4 · How wrong is it?

For each car, the **error** is the predicted MPG minus the actual MPG.

Negative numbers mean we under-predicted. Positive means we over-predicted.

We want one number that summarizes how wrong the whole model is.

**Why not just average the errors?** Because positives and negatives cancel.

If the model is over-predicting some cars and under-predicting others by the same amount, the average error is zero — but the model is still bad.

**The fix:** square the errors first. Squaring makes everything positive. Big errors get punished extra (squaring 10 is 100).

Now average those. That's the **Mean Squared Error** — MSE — the standard loss for regression.

Let's wrap it in a function so we can call it for any (w, b).

Same number. Now we can ask: for any line, how bad is it?

## Chapter 5 · MSE has a picture

Each squared error is literally the area of a square. Side length = the vertical distance from the line to the point. Area = that distance squared.

Plot the squares and the total gray area is the sum of squared errors. Divide by N for MSE.

The bigger the line's mistakes, the bigger the squares. Move the line, squares grow or shrink. **MSE going down = the squares getting smaller, on average.**

## Chapter 6 · Better guesses

Let's try a few different (w, b) pairs and see which one is smallest.

Manually tuning gets tedious fast. With two knobs, you could keep going — but real models have millions of parameters.

We need a way to find the best (w, b) automatically. That's gradient descent. But first, let's see what we're trying to navigate.

## Chapter 7 · The loss surface

MSE is a function of two inputs (w, b) and outputs one number. **Two inputs, one output. That's a surface.**

Compute MSE on a grid of (w, b) values.

Now plot it as a 3D surface.

**One bottom. No local pits. No tricks.** For linear regression, the bowl is always this clean — what we call *convex*.

Gradient descent walks downhill on this bowl. The bottom is the answer.

## Chapter 8 · The gradient

The **gradient** is the slope of the bowl at the current point. It tells us which direction is *uphill*.

For MSE, the math works out to:

$$\frac{\partial L}{\partial w} = \frac{2}{N}\sum_i x_i(w x_i + b - y_i)$$
$$\frac{\partial L}{\partial b} = \frac{2}{N}\sum_i (w x_i + b - y_i)$$

In code:

Compute the gradient at our guess (w=-3, b=30).

Both numbers are negative.

**Reading the gradient:** the gradient points *uphill*. Negative means uphill is in the negative direction. So *downhill* is in the *positive* direction — both `w` and `b` should increase to lower the loss.

But wait — we already saw that the true optimum has `w ≈ -5` (more negative, not less). The path to the optimum isn't straight — we'll see it curve.

## Chapter 9 · One step

Take a tiny step in the opposite direction of the gradient. The size of the step is the **learning rate** — usually written `η` (eta) or `lr`.

$$w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$$

MSE went down. One step worked. Now do it lots of times.

## Chapter 10 · The full descent loop

Same step, in a loop. Store the history so we can plot it later.

**About 10,000 iterations to fully converge.** That feels like a lot for two parameters. The reason: the MPG features aren't *normalized* — they range from 2.4 to 4.4. Gradient descent struggles when features are on different scales than each other or than the bias.

In real ML, you'd normalize the inputs (subtract mean, divide by std) and converge in a few hundred steps. Or use a smarter optimizer like Adam (PyTorch section below).

But this is gradient descent in its rawest form. Same algorithm from calculus Ep 17.

Plot the loss curve to see the descent.

Steep drop, then a long flattening tail. Classic convergence shape.

## Chapter 11 · The descent path on the bowl

Overlay the path on the loss surface from Chapter 7. The ball rolls down.

Pink dot is where we started. Green dot is where we ended up. The line connecting them is the descent path. **That path is what training looks like.**

## Chapter 12 · The fitted line

Plot the trained model over the data and compare to where we started.

The yellow line is the trained model. Tighter fit through the data than the pink dashed starting guess. **By-hand linear regression — done.**

## Chapter 13 · The same thing in PyTorch — the 5-line loop

PyTorch handles the gradient for us. `loss.backward()` computes ∂L/∂w and ∂L/∂b automatically (chain rule, from calculus Ep 15-18). The five lines below are the same loop we just wrote by hand.

Convert data to tensors. Shape `(N, 1)` for the linear layer.

Set up the model, optimizer, and loss.

Why **Adam** instead of vanilla SGD? Adam adapts the learning rate per parameter, which handles un-normalized features gracefully. SGD would also work but need 10,000+ iterations like our by-hand version. Adam takes a few thousand.

The five lines:

Pull out the trained parameters and compare.

Same answer (within numerical precision). Different code path, same destination.

## Chapter 14 · The same thing in sklearn — 3 lines

Sklearn wraps the whole training loop. For most projects, this is what you'd actually write.

Read out the trained parameters.

All three agree.

Three different code paths, one true answer. The math doesn't care how you got there.

Predict the MPG of a brand-new 3,500-pound car.

About 17.9 MPG. Reasonable — it sits right in the middle of our training data range.

## Chapter 15 · More features

Real cars have more than just weight. Let's add horsepower and engine displacement.

Three features per car instead of one. The model becomes:

$$y = b + w_1 \cdot \text{weight} + w_2 \cdot \text{horsepower} + w_3 \cdot \text{displacement}$$

*Same training loop. One extra weight per feature. That's it.*

Read the coefficients — one per feature.

Each coefficient says: **holding the other features constant, this is how much MPG changes per unit of this feature.**

All three are negative — heavier, more powerful, larger-engined cars get fewer MPG. Matches intuition.

## Chapter 16 · What we did

1. Loaded a dataset and plotted it.
2. Defined a model `y = wx + b` and a loss `MSE`.
3. Visualized MSE as squared distances, then as a 3D bowl over (w, b).
4. Computed the gradient by hand. Took one step. Looped 10,000 times.
5. Plotted the descent path on the bowl.
6. Did the same thing in PyTorch (5 lines) and sklearn (3 lines). All three converged to the same answer.
7. Extended to multiple features. Same loop. Same algorithm.

**Next notebook: logistic regression.** Same training loop with two substitutions — sigmoid in the forward pass, log loss in place of MSE.